# Inspect the analysis layers
Quick visual checks of any derived raster on the 90 m grid. Run the pipeline up to `make features` first.
`zoom()` crops by lon/lat and plots at full resolution — use it to look for tile seams, nodata gaps and artifacts.

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, geopandas as gpd, pandas as pd
from ecoregions.config import load_config, Paths
from ecoregions.grid import Grid, read
from ecoregions.qa import zoom
cfg = load_config("../config.yaml"); paths = Paths(cfg); grid = Grid.load(paths.grid_json)

In [ ]:
# list what is available
sorted(p.stem for p in paths.interim.glob("*.tif"))

In [ ]:
# full-resolution crop: the Salt Range escarpment
dem = read(paths.raster("dem"))
zoom(dem, grid, (72.4, 32.5, 73.2, 32.95), paths.qa / "nb_dem_saltrange.png", "DEM — Salt Range", cmap="terrain")
plt.imshow(plt.imread(paths.qa / "nb_dem_saltrange.png")); plt.axis("off");

In [ ]:
# nodata audit within the study area
mask = read(paths.raster("mask")) == 1
for name in ["dem", "slope", "hand", "soil_clay", "clim_bio12", "geo_litho"]:
    a = read(paths.raster(name))
    bad = np.isnan(a) if a.dtype.kind == "f" else (a == 0)
    print(f"{name:12s} nodata in study area: {100 * (bad & mask).sum() / mask.sum():.2f}%")